In [3]:
from Utils import process_and_route_single,generate_summary_single
import h5py

import torch

In [2]:
a = h5py.File('Data/Metadata/mr_hisum_metadata.h5')

In [3]:
b = h5py.File('Data/Metadata/videoxum_metadata.h5')

In [5]:
batch = b['video_160']

In [48]:
gtscore_xum = batch['gtscore_xum'][...]
positions = batch['positions'][...]
user_summary = batch['user_summary'][...]

# Tesst eval strats

In [47]:
batch['gtscore_xum'][...].shape

(10, 83)

In [4]:
from Utils.post_process import knapSack

In [5]:
import numpy as np 

In [ ]:
def generate_summary_no_upsample(score,shot_bounds):
    if len(score) != shot_bounds[-1][-1]+1: 
        positions = np.arange(0,len(score))
        step = 5
        shot_bounds = np.array([(start,min(start+step,len(score))) for start in range(0,len(score),step)])

    frame_scores = score
    shot_imp_scores = []
    shot_lengths = []
    for shot in shot_bounds:
        shot_lengths.append(shot[1] - shot[0] + 1)
        shot_imp_scores.append((frame_scores[shot[0]:shot[1] + 1].mean()).item())
    # Select the best shots using the knapsack implementation
    final_shot = shot_bounds[-1]
    final_max_length = int((final_shot[1] + 1) * 0.15)
    selected = knapSack(final_max_length, shot_lengths, shot_imp_scores, len(shot_lengths))
    summary = np.zeros(final_shot[1] + 1, dtype=np.int8)
    for shot in selected:
        summary[shot_bounds[shot][0]:shot_bounds[shot][1] + 1] = 1
    return summary

In [27]:
generate_summary_no_upsample(batch['gtscore'][...],batch['shot_bounds'][...])

154
154


array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0], dtype=int8)

In [32]:
batch['shot_bounds'][...]

array([[  0,   1],
       [  1,   2],
       [  2,   3],
       [  3,   6],
       [  6,   8],
       [  8,  10],
       [ 10,  15],
       [ 15,  35],
       [ 35,  36],
       [ 36,  40],
       [ 40,  42],
       [ 42,  61],
       [ 61,  64],
       [ 64,  65],
       [ 65,  66],
       [ 66,  67],
       [ 67,  73],
       [ 73,  84],
       [ 84,  86],
       [ 86,  89],
       [ 89, 105],
       [105, 107],
       [107, 109],
       [109, 111],
       [111, 113],
       [113, 115],
       [115, 121],
       [121, 122],
       [122, 126],
       [126, 128],
       [128, 130],
       [130, 142],
       [142, 144],
       [144, 146],
       [146, 149],
       [149, 151],
       [151, 153],
       [153, 164],
       [164, 175],
       [175, 179],
       [179, 188],
       [188, 197],
       [197, 198],
       [198, 200],
       [200, 216],
       [216, 232],
       [232, 233],
       [233, 236],
       [236, 243],
       [243, 245],
       [245, 256],
       [256, 257]], dtype=int64

In [24]:
len(batch['user_summary'][...][None,:])

1

In [30]:
len(batch['user_summary'][...].shape)

1

In [25]:
for summary in batch['user_summary'][...][None,:]:
    print(summary)

[0. 0. 0. 0. 0. 0. 1. 1. 1. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 1. 1. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 1. 1. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 1. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]


In [6]:
metadata_dict = {'shot_bounds': batch['shot_bounds'][...],'positions': batch['positions'][...][:-1],'n_frames':batch['n_frames'][()]}
ground_truth_dict = {'user_summary':batch['user_summary'][...]} 

SyntaxError: invalid syntax (1854261830.py, line 1)

In [56]:
batch['positions'][...]

array([  0,   3,   6,   9,  12,  15,  18,  21,  24,  27,  30,  33,  36,
        39,  42,  45,  48,  51,  54,  57,  60,  63,  66,  69,  72,  75,
        78,  81,  84,  87,  90,  93,  96,  99, 102, 105, 108, 111, 114,
       117, 120, 123, 126, 129, 132, 135, 138, 141, 144, 147, 150, 153,
       156, 159, 162, 165, 168, 171, 174, 177, 180, 183, 186, 189, 192,
       195, 198, 201, 204, 207, 210, 213, 216, 219, 222, 225, 228, 231,
       234, 237, 240, 243, 246, 249, 252, 255, 258, 261, 264, 267, 270,
       273, 276, 279, 282, 285, 288, 291, 294, 297, 300, 303, 306, 309,
       312, 315, 318, 321, 324, 327, 330, 333, 336, 339, 342, 345, 348,
       351, 354, 357, 360, 363, 366, 369, 372, 375, 378, 381, 384, 387,
       390, 393, 396, 399, 402, 405, 408, 411, 414, 417, 420, 423, 426,
       429, 432, 435, 438, 441, 444, 447, 450, 453, 456, 459, 462],
      dtype=int64)

In [4]:
import numpy as np

In [9]:
from Utils.post_process import knapSack
def generate_summary_single(shot_bound,score,n_frames,positions,return_shot_info = False):
    frame_init_scores = score
    frame_scores = np.zeros(n_frames, dtype=np.float32)
    if positions.dtype != int:
        positions = positions.astype(np.int32)
    
    if positions[-1] != n_frames:
        positions = np.concatenate([positions, [n_frames]])

    for i in range(len(positions) - 1):
        pos_left, pos_right = positions[i], positions[i + 1]
        if i == len(frame_init_scores):
            frame_scores[pos_left:pos_right] = 0
        else:
            frame_scores[pos_left:pos_right] = frame_init_scores[i]

    # Compute shot-level importance scores by taking the average importance scores of all frames in the shot
    shot_imp_scores = []
    shot_lengths = []
    for shot in shot_bound:
        shot_lengths.append(shot[1] - shot[0] + 1)
        shot_imp_scores.append((frame_scores[shot[0]:shot[1] + 1].mean()).item())

    # Select the best shots using the knapsack implementation
    final_shot = shot_bound[-1]
    final_max_length = int((final_shot[1] + 1) * 0.15)

    selected = knapSack(final_max_length, shot_lengths, shot_imp_scores, len(shot_lengths))

    # Select all frames from each selected shot (by setting their value in the summary vector to 1)
    final_length = final_shot[1] if final_shot[1] == n_frames else final_shot[1] +(n_frames-final_shot[1])
    summary = np.zeros(final_length, dtype=np.int8)
    for shot in selected:
        summary[shot_bound[shot][0]:shot_bound[shot][1] + 1] = 1
    if return_shot_info:
        return shot_lengths, shot_imp_scores,selected,summary
    return summary

In [12]:
out = generate_summary_single(metadata_dict['shot_bounds'],batch['gtscore'][...],metadata_dict['n_frames'],metadata_dict['positions'])

In [13]:
out.shape

(463,)

In [40]:
metadata_dict['n_frames']

463

In [5]:
c = h5py.File('Data\Metadata/tvsum_metadata.h5')

In [6]:
batch_tv = c['video_1']

In [11]:
batch_tv.keys()

<KeysViewHDF5 ['n_frames', 'positions', 'shot_bounds', 'user_score', 'user_summary']>

In [14]:
batch_tv['user_score'][...].mean(0).shape

(10597,)

In [16]:
sum = generate_summary_single(batch_tv['shot_bounds'][...],batch_tv['user_score'][...].mean(0),
                        batch_tv['n_frames'][()],positions = batch_tv['positions'][...])

In [17]:
len(sum)

10597

In [19]:
batch_tv['user_summary'][...].shape

(20, 10597)

In [57]:
batch_tv['shot_bounds'][...][[-1]]

array([[ 9600, 10596]], dtype=int64)

In [59]:
batch_tv['n_frames'][()]

10597

In [ ]:
len(batch_tv['sh'][...])

707

In [49]:
shot_bounds = np.array([(start,min(start+5,metadata_dict['n_frames'])) for start in range(0,metadata_dict['n_frames'],5)])

In [2]:
from Models.losses.loss_functions import *

In [ ]:
loss_creation_dict = {'mse':{}}

In [3]:
MeanSquaredError(**{})

In [2]:
test_1 = WeightedLossStack()

TypeError: __init__() missing 2 required positional arguments: 'losses' and 'weights'

In [55]:
positions

NameError: name 'positions' is not defined

In [42]:
batch['shot_bounds'][...]

array([[  0,  30],
       [ 30,  60],
       [ 60,  90],
       [ 90, 120],
       [120, 150],
       [150, 180],
       [180, 210],
       [210, 240],
       [240, 270],
       [270, 300],
       [300, 330],
       [330, 360],
       [360, 390],
       [390, 420],
       [420, 450],
       [450, 463]], dtype=int64)

In [39]:
ground_truth_dict['user_summary'].shape

(10, 463)

In [32]:
process_and_route_single(torch.from_numpy(batch['gtscore'][...]),torch.from_numpy(batch['gtscore'][...]),metadata_dict,ground_truth_dict,eval_type= 'avg',post_process = 'summary_gen',metric='f1')

ValueError: Found input variables with inconsistent numbers of samples: [463, 464]